# Support Vector Machine Regression

Support Vector Machines are a powerful class of algorithms originally designed for classification, but adapted here for regression (SVR - Support Vector Regression). Unlike linear regression which minimizes the sum of squared errors across all points, SVR tries to fit a tube of width $\epsilon$ around the data and only penalizes points that fall outside this tube.

As noted in [CMOR 438 Lecture 7.1](../../Data_Science_and_Machine_Learning_Spring_2022/Lecture_7/Lecture_7_1.ipynb), SVMs are classified as a nonparametric algorithm alongside KNN and Decision Trees, meaning they do not assume a fixed number of parameters - the model complexity grows with the data.

SVR uses the **kernel trick** to project data into higher-dimensional spaces where linear separation (or in this case, linear regression) becomes possible. We will test four kernels:
- **Linear**: No transformation, equivalent to linear regression with an $\epsilon$-insensitive loss
- **Polynomial**: Maps features to polynomial combinations
- **RBF (Radial Basis Function)**: Maps to infinite-dimensional space, captures local patterns
- **Sigmoid**: Similar to neural network activation, captures S-shaped relationships

We continue using the [California Housing Dataset](https://www.kaggle.com/datasets/camnugent/california-housing-prices).

---

## Data Preprocessing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn as sk
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

data_df = pd.read_csv("housing.csv")
data_df['total_bedrooms'] = data_df['total_bedrooms'].fillna(data_df['total_bedrooms'].median())
data_encoded = pd.get_dummies(data_df, columns=['ocean_proximity'], dtype=float)

data = data_encoded.to_numpy()
target_col = data_encoded.columns.get_loc('median_house_value')
feature_cols = [i for i in range(data.shape[1]) if i != target_col]
x_data = data[:, feature_cols]
y = data[:, target_col]
feature_names = [data_encoded.columns[i] for i in feature_cols]

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_data)

X_train, X_test, y_train, y_test = train_test_split(
    x_scaled, y, test_size=0.33, shuffle=True, random_state=42
)

print(f"Training: {X_train.shape}, Test: {X_test.shape}")

---

## Kernel Comparison

We train SVR models with four different kernels and compare their performance. SVR is computationally expensive on large datasets, so we use a subsample for training to keep runtimes manageable.

In [ ]:
np.random.seed(42)
sample_size = 5000
sample_idx = np.random.choice(X_train.shape[0], sample_size, replace=False)
X_train_sample = X_train[sample_idx]
y_train_sample = y_train[sample_idx]

kernels = ['linear', 'poly', 'rbf', 'sigmoid']
models = {}
predictions = {}

for kernel in kernels:
    print(f"Training SVR with {kernel} kernel...")
    svr = SVR(kernel=kernel, C=100000, epsilon=10000)
    svr.fit(X_train_sample, y_train_sample)
    models[kernel] = svr
    predictions[kernel] = svr.predict(X_test)
    print(f"  Done.")

print("\nAll models trained.")

The `C` parameter controls the penalty for points outside the $\epsilon$-tube. A larger C means the model tries harder to fit every point (risking overfitting), while a smaller C allows more slack (risking underfitting). The `epsilon` parameter defines the width of the tube within which no penalty is applied.

---

## Results Visualization

In [ ]:
fig, axs = plt.subplots(4, 1, figsize=(18, 28))

for i, kernel in enumerate(kernels):
    y_pred = predictions[kernel]

    mse = np.round((1 / X_test.shape[0]) * np.sum((y_pred - y_test) ** 2), 2)
    mae = np.round((1 / X_test.shape[0]) * np.sum(np.abs(y_pred - y_test)), 2)
    r2 = np.round(sk.metrics.r2_score(y_test, y_pred), 4)

    axs[i].scatter(range(X_test.shape[0]), y_pred, s=5, alpha=0.3, label='Predicted')
    axs[i].scatter(range(X_test.shape[0]), y_test, s=5, alpha=0.3, label='Actual')
    axs[i].set_title(f'SVR - {kernel.upper()} Kernel', fontsize=16)
    axs[i].set_xlabel('Sample Number', fontsize=12)
    axs[i].set_ylabel('House Value ($)', fontsize=12)
    axs[i].legend(fontsize=10)

    errors = f"MSE: {mse}\nMAE: {mae}\nR2: {r2}"
    axs[i].text(0.01, 0.95, errors, transform=axs[i].transAxes, verticalalignment='top',
               fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

The kernel choice significantly impacts SVR performance:
- **RBF** typically performs best as it can capture nonlinear local patterns in the data
- **Linear** performs similarly to standard linear regression since it applies the same linear model with a different loss function
- **Polynomial** can capture polynomial relationships but may overfit or underfit depending on the degree
- **Sigmoid** often performs worst as its shape doesn't naturally match housing price distributions

---

## Feature Importance via Prediction Analysis

We can visualize how the best-performing SVR model's predictions relate to key features.

In [ ]:
best_kernel = max(predictions.keys(), key=lambda k: sk.metrics.r2_score(y_test, predictions[k]))
best_pred = predictions[best_kernel]
print(f"Best kernel: {best_kernel}")

plot_features = ['median_income', 'housing_median_age', 'latitude', 'longitude']
plot_indices = [feature_names.index(f) for f in plot_features]

fig, axs = plt.subplots(2, 2, figsize=(20, 16))
fig.suptitle(f"SVR ({best_kernel.upper()}) - Predicted vs Actual by Feature", fontsize=24)

for idx, (feat, col_idx) in enumerate(zip(plot_features, plot_indices)):
    row, col = idx // 2, idx % 2
    axs[row, col].scatter(X_test[:, col_idx], y_test, alpha=0.15, s=5, label='Actual')
    axs[row, col].scatter(X_test[:, col_idx], best_pred, alpha=0.15, s=5, label='Predicted', color='coral')
    axs[row, col].set_xlabel(feat, fontsize=14)
    axs[row, col].set_ylabel('median_house_value', fontsize=14)
    axs[row, col].legend(fontsize=12)

plt.tight_layout()
plt.show()

---

## Error Distribution Comparison

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(18, 14))

for i, kernel in enumerate(kernels):
    row, col = i // 2, i % 2
    errors = predictions[kernel] - y_test
    axs[row, col].hist(errors, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    axs[row, col].set_title(f'{kernel.upper()} Kernel Error Distribution', fontsize=14)
    axs[row, col].set_xlabel('Prediction Error ($)', fontsize=12)
    axs[row, col].set_ylabel('Frequency', fontsize=12)
    axs[row, col].axvline(x=0, color='red', linestyle='--', linewidth=2)

plt.tight_layout()
plt.show()

---

## 3D Geographic Visualization

In [ ]:
long_idx = feature_names.index('longitude')
lat_idx = feature_names.index('latitude')

fig = plt.figure(figsize=(16, 10))
ax = fig.add_subplot(projection='3d')

ax.scatter3D(X_test[:, long_idx], X_test[:, lat_idx], best_pred,
             c=best_pred, cmap='Reds', alpha=0.3, s=5, label='Predicted')
ax.scatter3D(X_test[:, long_idx], X_test[:, lat_idx], y_test,
             c=y_test, cmap='Greens', alpha=0.3, s=5, label='Actual')

ax.set_xlabel('Longitude (normalized)')
ax.set_ylabel('Latitude (normalized)')
ax.set_zlabel('Median House Value ($)')
ax.set_title(f'SVR ({best_kernel.upper()}) - House Value vs Location', fontsize=18)
ax.legend(fontsize=14)
ax.view_init(20, 30)
plt.tight_layout()
plt.show()

Compared to linear regression's 3D plot, SVR (particularly with the RBF kernel) can capture more of the nonlinear geographic price patterns. The predicted surface is less smooth and more adaptive to local price variations.

---

## Summary

SVR offers a different approach to regression compared to gradient descent-based methods:
- The **RBF kernel** typically outperforms other kernels by capturing nonlinear patterns
- The **linear kernel** performs comparably to standard linear regression
- SVR is **computationally expensive** - we had to subsample the training data, which limits its effectiveness on this 20K+ sample dataset
- The $\epsilon$-tube loss function makes SVR more robust to outliers than standard MSE-based regression

For large datasets like California Housing, tree-based methods (Random Forests, Gradient Boosting) are generally preferred over SVR due to their better scalability and comparable or superior accuracy.